In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

df_silver = spark.table("taxi_catalog.silver.yellow_taxi_clean")


print("total_rows:",df_silver.count())
print("total_vendor:",df_silver.select("VendorID").distinct().count())
print("total_payment_type:",df_silver.select("payment_type").distinct().count())
print("total_ratecode:",df_silver.select("RatecodeID").distinct().count())

display(df_silver.limit(10))

Monthly aggregation


In [0]:
df_monthly = (
    df_silver
    .groupBy("pickup_year", "pickup_month")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_duration_minutes").alias("avg_trip_duration"),
        F.avg("passenger_count").alias("avg_passengers")
    )
    .orderBy("pickup_year", "pickup_month")
)

display(df_monthly)

gold schema 


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS taxi_catalog.gold;

save montly data 


In [0]:
df_monthly.write.format("delta").mode("overwrite").saveAsTable("taxi_catalog.gold.monthly_taxi_metrics")

In [0]:
display(spark.table("taxi_catalog.gold.monthly_taxi_metrics"))

Aggregation


In [0]:
df_payment = (
    df_silver
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_transaction_amount")
    )
    .orderBy("payment_type")
)

display(df_payment)

save aggregation data 


In [0]:
df_payment.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("taxi_catalog.gold.payment_metrics")

In [0]:
display(spark.table("taxi_catalog.gold.payment_metrics"))

Top pickup locations

In [0]:
df_top_pickup = (
    df_silver
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue")
    )
    .orderBy(F.desc("total_trips"))
    .limit(10)
)

display(df_top_pickup)

In [0]:
df_top_pickup.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("taxi_catalog.gold.top_pickup_locations")

In [0]:
display(spark.table("taxi_catalog.gold.top_pickup_locations"))